# 单侧年度超额目标训练
仅把年度标准差改为负超额RMS，系数仍0.50；原结构、全训练史及完整风控。3次拟合3新账户，不扫描参数。

In [1]:
from pathlib import Path
import os,json,qlib
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/annual_shortfall_training_20261015'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us')
from src.workflow.paired_allocator_experiment import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
print(summary)

# 单侧年度超额目标：训练回测不晋升

完成3次真实新拟合、3套新增Qlib原生账户，复用3个冻结原结构对照模型和9套参考账户。使用原全成熟历史、原输入、原网络、原3个seed、300轮末轮等权集成；唯一变化是将年度超额标准差惩罚替换为年度负超额均方根，系数保持0.50。原目标并非代码错误，此实验仅检验目标语义差异，未增加市场信息。

原损失项为 -mean(a)+0.50*std(a)，候选为 -mean(a)+0.50*sqrt(mean(relu(-a)^2))，a是各年日均对数超额。候选不会因已跑赢年度进一步改善而增加单侧惩罚。零负超额时梯度有限；这个数学性质不等于真实账户能产生alpha。训练日期/成熟时点、缩放器、初始网络及冻结对照预测逐值检查通过。

回测2023-01-03信号至2026-10-09退出、944区间；10万美元，冻结IBKR Pro Tiered费表投影，独立0/2/5bp账户，原开盘路由/恢复/风险投影和30%回撤门保留。费表是统一研究投影，不代表逐年实际券商账单。历史反复使用，不是新的独立留出。

|账户|0bp净收益|2bp净收益|5bp净收益|
|---|---:|---:|---:|
|原目标配对对照|280.24%|269.16%|266.08%|
|单侧年度目标|205.52%|239.93%|156.47%|
|SOXX|387.75%|387.73%|387.70%|

2bp监控最大回撤26.97%，CVaR与回撤门通过；滚动126日正超额占比28.45%，年度/区块置信/净超额门失败。实际平均现金54.00%，高于对照46.43%；不是通过减少现金获得更高收益。不同成本独立风险状态导致收益非单调，不可挑一个成本宣布有效。候选三成本均弱于配对原模型，也弱于此前长期小配置头研究参考（后者2bp335.36%）；不以较弱对照宣称进步。

结论not_promoted_annual_shortfall_objective；没有证据支持仅将年度离散惩罚改为单侧短缺即可解决现金配置和超额问题。关闭固定路线，不调惩罚系数、训练轮次、seed或风险预算挽救。生产未替换，稳定超额仍未完成。下一需要仍是训练期OOF可重复的独立预测信息；不再以目标语义更贴近需求推断必然有效。

完整训练模型、预测、来源及原生账本存Qlib Recorder与NAS；逐对象冷恢复哈希通过，6项针对性测试通过（1项新增单侧目标梯度合同，5项既有模型/分钟/账户合同）。价格代理、实际成交时延、全分钟最大回撤和真实派息认证边界保持不变。

训练主样本3014条，2011-01-03至2022-12-21；辅助20741条，最后标签成熟于2022-12-30 09:31:00-05:00。

2bp年度候选 / SOXX收益：2023：40.02% / 56.60%；2024：29.59% / 20.97%；2025：25.24% / 46.80%；2026：49.59% / 75.38%。2026为不完整年度。
